In [62]:
from deepface import DeepFace
import os
import numpy as np
from IPython.display import Image
import pickle


In [63]:
def encoding_for_celeb_images (dir_path_of_images):
    EMBEDDINGS_ON_DISK = "persisted_embeddings.pkl"
    CELEFILENAMES_ON_DISK = "persisted_image_names.pkl"
    if (os.path.exists(EMBEDDINGS_ON_DISK) and os.path.exists(CELEFILENAMES_ON_DISK)):
        with open(EMBEDDINGS_ON_DISK, "rb") as f:
            embedding_dict = pickle.load(f)
        with open(CELEFILENAMES_ON_DISK, "rb") as f:
            celeb_file_names = pickle.load(f)
        return embedding_dict, celeb_file_names
    known_encodings =[]
    known_images = []
    for file in os.listdir(dir_path_of_images):
        #fsdecode function decode the file into filename
        filename = os.fsdecode(file)
        # for now ignoring filenames that have non ascii characters. This needs to be fixed
        if filename.isascii():
            known_encodings.append (DeepFace.represent(img_path=dir_path_of_images+"/"+filename, model_name="ArcFace", enforce_detection=False)[0]['embedding'])
            known_images.append(filename)
            with open(EMBEDDINGS_ON_DISK, "wb") as f:
                pickle.dump(known_encodings, f)
            with open(CELEFILENAMES_ON_DISK, "wb") as f:
                pickle.dump(known_images, f)
    return (known_encodings,known_images)



In [64]:
import math
def find_matching_face (known_encodings, known_images, path_of_image_to_match):
    encoding_to_match = DeepFace.represent(img_path=path_of_image_to_match, model_name="ArcFace")[0]['embedding']
    min_euclidean_dis = float ("inf")
    matched_celeb =""
    for i in range (len(known_encodings)):
        #find euclidean dis between known images and test image
        test_euclidean_dist_square =0
        for known_vector_dimesion, test_vect_dimension in zip (known_encodings[i], encoding_to_match):
            test_euclidean_dist_square += (known_vector_dimesion - test_vect_dimension)**2
        sqrt_equlidian_dist = math.sqrt(test_euclidean_dist_square)
        if min_euclidean_dis > sqrt_equlidian_dist:
            min_euclidean_dis = sqrt_equlidian_dist
            matched_celeb = known_images[i]
        
    return matched_celeb


In [ ]:
encodings, images = encoding_for_celeb_images ("/Users/sourabh/software_development/env_jupyter_spark_vsc/iitr_pgcp/ImdbImagesCelebs")

find_matching_face (encodings, images,"./myimage.jpg")


'George Burns.jpg'